In [ ]:
import sys
import os
from google.colab import userdata

# 1. 自动从 Colab Secrets 获取你存入的 Token
GIT_TOKEN = userdata.get('GITHUB_TOKEN')

# 2. 根据你的截图填入对应的用户名和仓库名
GIT_USER = "Sheng-Pan"
GIT_REPO = "Decentralized-federated-learning2"

# 3. 构建带认证信息的 URL
repo_url = f"https://{GIT_TOKEN}@github.com/{GIT_USER}/{GIT_REPO}.git"

# 4. 克隆仓库 (如果文件夹已存在则跳过，防止报错)
if not os.path.exists(GIT_REPO):
    !git clone {repo_url}
else:
    print(f"{GIT_REPO} already exists.")

# 5. 将仓库路径添加到系统路径，以便 Python 找到 functions2.py
repo_path = os.path.join("/content", GIT_REPO)
if repo_path not in sys.path:
    sys.path.append(repo_path)


Cloning into 'Decentralized-federated-learning2'...
remote: Enumerating objects: 39, done.
remote: Counting objects: 100% (39/39), done.
remote: Compressing objects: 100% (39/39), done.
remote: Total 39 (delta 19), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (39/39), 138.89 KiB | 1.13 MiB/s, done.
Resolving deltas: 100% (19/19), done.


In [ ]:
# 4. 强制更新仓库 (如果存在则拉取最新，不存在则克隆)
if not os.path.exists(GIT_REPO):
    print("Cloning new repository...")
    !git clone {repo_url}
else:
    print(f"{GIT_REPO} already exists. Pulling latest changes...")
    # 切换进目录更新，然后再切换出来
    %cd {GIT_REPO}
    !git pull
    %cd ..

# 5. 关键的一步：强制重新加载已经导入的模块
import importlib
import defense
importlib.reload(defense)
import backdoor
importlib.reload(backdoor)
import MAB_fun
importlib.reload(MAB_fun)
import trainer
importlib.reload(trainer)
import eval_DFL
importlib.reload(eval_DFL)
import main_cnn_GPU
importlib.reload(main_cnn_GPU)

Decentralized-federated-learning2 already exists. Pulling latest changes...
/content/Decentralized-federated-learning2
remote: Enumerating objects: 5, done.
remote: Counting objects: 100% (5/5), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 3 (delta 2), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (3/3), 929 bytes | 929.00 KiB/s, done.
From https://github.com/Sheng-Pan/Decentralized-federated-learning2
   fcf0b00..645d09d  main       -> origin/main
Updating fcf0b00..645d09d
Fast-forward
 eval_DFL.py | 2 +-
 1 file changed, 1 insertion(+), 1 deletion(-)
/content


<module 'main_cnn_GPU' from '/content/Decentralized-federated-learning2/main_cnn_GPU.py'>

In [ ]:

import os
import time
import pandas as pd
import sys

from datetime import datetime
import pytz

from data_loader import allocate_malicious_nodes
from data_loader import generate_topology
from main_cnn_GPU import run_simulation_CNN_GPU
from data_loader import set_seed
from data_loader import distribute_data
from data_loader import get_data
from theoretical_intensity import calculate_theoretical_intensity
from google.colab import userdata
import copy
from torch.utils.data import DataLoader
from huggingface_hub import HfApi, snapshot_download


original_stdout = sys.stdout
original_stderr = sys.stderr
class DualLogger(object):
    def __init__(self, file_path):
        self.terminal = sys.stdout
        self.log = open(file_path, "a", encoding='utf-8')

    def write(self, message):
        self.terminal.write(message)
        self.log.write(message)
        self.log.flush()

    def flush(self):
        self.terminal.flush()
        self.log.flush()

    def close(self):
        self.log.close()

import sys

NORM_FACTOR = 1
def upload_with_retry(path, repo_path, max_retries=3):
    for i in range(max_retries):
        try:
            api.upload_file(
                path_or_fileobj=path,
                path_in_repo=repo_path,
                repo_id=REPO_ID,
                repo_type="dataset"
            )
            print(f"✅ 上传成功: {os.path.basename(path)}")
            return True
        except Exception as upload_err:
            print(f"⚠️ 第 {i+1} 次上传失败 ({os.path.basename(path)}): {upload_err}")
            if i < max_retries - 1:
                time.sleep(5)  # 等待5秒后重试
            else:
                print(f"❌ 最终上传放弃: {os.path.basename(path)}")
                return False
HF_TOKEN = userdata.get('HF_TOKEN')
REPO_ID = "JONESMITH007/DFL"
LOCAL_ROOT = "./DFL"
api = HfApi(token=HF_TOKEN)
SAVE_PATH = os.path.join(LOCAL_ROOT, "FL_Experiments/cnn_Results_2026_inew")
os.makedirs(SAVE_PATH, exist_ok=True)
# ==========================================
# 1.Experiment settup
# ==========================================



NUM_CLIENTS = 20
BOOST_FACTORS = 2
MALICIOUS_RATIO = 0.3
defense_budget  = int(NUM_CLIENTS * 0.2)
GLOBAL_ROUNDS = 15
EPOCHS = 5
ATK_TYPE = 'neurotoxin'
INTENSITY = 4.0

SCALE_FACTOR = 0.8
train_ds, test_ds = get_data()
test_loader = DataLoader(test_ds, batch_size=256)


SEEDS = [1,2,3]
MECHANISM_LIST = ['MAB',  'FLAME','CosL2', 'FedAvg','TrimmedMean', 'Krum']
MAL_RATIOS = [ 0.3, 0.1, 0.2]
TOPO_TYPES = [ 'scale_free']
DEFENSE_RATIOS = [ 0.2]

NUM_CLIENTS = 20
GLOBAL_ROUNDS = 15
NORM_FACTOR = 1
SCALE_FACTOR = 0.8
ATK_TYPE = 'neurotoxin'
INTENSITY = 4.0




for topo_type in TOPO_TYPES:
    for mal_ratio in MAL_RATIOS:
        for def_ratio in DEFENSE_RATIOS:


            num_mal = int(NUM_CLIENTS * mal_ratio)

            current_def_budget = int(NUM_CLIENTS * def_ratio)
            for seed_val in SEEDS:


                print(f"\n{'='*60}")
                print(f"⏰ : {time.strftime('%Y-%m-%d %H:%M:%S')}")
                print(f"📡 : Topo={topo_type}, Mal={mal_ratio}, Def={def_ratio}")
                print(f"{'='*60}")
                print(f"\n{'#'*60}")
                print(f"📡 Topo: {topo_type} | Mal: {mal_ratio} | Def: {def_ratio} | Seed: {seed_val}")
                print(f"{'#'*60}")

                set_seed(seed_val)
                client_datasets = distribute_data(train_ds, NUM_CLIENTS)
                G = generate_topology(NUM_CLIENTS, topo_type)
                neighbors = {node: list(G.neighbors(node)) for node in G.nodes()}


                malicious_clients, defense_nodes = allocate_malicious_nodes(
                    G, num_mal, current_def_budget, topology_type=topo_type,placement='Topology-Aware',# placement='Topology-Aware'
                )

                theo_intensities = calculate_theoretical_intensity(
                    neighbors, malicious_clients, NUM_CLIENTS, BOOST_FACTORS, lambda_benign=0.3
                )

                for mech in MECHANISM_LIST:
                    all_results = []
                    csv_filename = f"Final_CNN_{topo_type}_MR{mal_ratio}_DR{def_ratio}_{mech}_seed{seed_val}.csv"
                    full_save_path = os.path.join(SAVE_PATH, csv_filename)
                    log_filename = f"Log_CNN_{topo_type}_MR{mal_ratio}_DR{def_ratio}_{mech}_seed{seed_val}.txt"
                    log_full_path = os.path.join(SAVE_PATH, log_filename)

                    if os.path.exists(full_save_path):
                        print(f"⏩Eisting file: {csv_filename}")
                        continue
                    logger = DualLogger(log_full_path)
                    sys.stdout = logger
                    sys.stderr = logger
                    shanghai_tz = pytz.timezone('Asia/Shanghai')
                    print(f"🚀 Running: {mech}")
                    start_wall_time = datetime.now(shanghai_tz).strftime("%Y-%m-%d %H:%M:%S")
                    start_tick = time.time()

                    _, _, accs, asrs = run_simulation_CNN_GPU(
                        seed_val, NUM_CLIENTS, defense_nodes, malicious_clients,
                        G, neighbors, client_datasets, test_loader,
                        atk_type=ATK_TYPE,
                        mechanism=mech,
                        intensity=INTENSITY,
                        norm_factor=NORM_FACTOR,
                        debug_mode=False,
                        GLOBAL_ROUNDS=GLOBAL_ROUNDS,
                        epochs=5, debug=False
                    )
                    end_tick = time.time()
                    duration_sec = round(end_tick - start_tick, 2)

                    result_entry_base = {
                        'seed': seed_val,
                        'mechanism': mech,
                        'malicious_ratio': mal_ratio,
                        'defense_ratio': def_ratio,
                        'topology': topo_type,
                        'norm_factor': NORM_FACTOR,
                        'scale_factor': SCALE_FACTOR,
                        'global_rounds': GLOBAL_ROUNDS,
                        'start_time': start_wall_time,
                        'duration_sec': duration_sec
                    }

                    for i in range(NUM_CLIENTS):
                        client_row = copy.deepcopy(result_entry_base)
                        client_row.update({
                            'client_id': i,
                            'final_acc': accs[i],
                            'final_asr': asrs[i],
                            'theo_intensity': theo_intensities[i] if i < len(theo_intensities) else 0.0,
                            'node_type': 'MAL' if i in malicious_clients else ('DEF' if i in defense_nodes else 'BEN')
                        })
                        all_results.append(client_row)

                    pd.DataFrame(all_results).to_csv(full_save_path, index=False)


                    sys.stdout = original_stdout

                    csv_repo_path = f"FL_Experiments/cnn_Results_2026_inew/{csv_filename}"
                    log_repo_path = f"FL_Experiments/cnn_Results_2026_inew/{log_filename}"


                    upload_with_retry(full_save_path, csv_repo_path)
                    upload_with_retry(log_full_path, log_repo_path)

                    sys.stdout = logger

print(f"\n🎉 Experiments compeleted！")

100%|██████████| 187M/187M [00:10<00:00, 17.1MB/s]
100%|██████████| 89.0M/89.0M [00:05<00:00, 16.5MB/s]
100%|██████████| 99.6k/99.6k [00:00<00:00, 210kB/s]



⏰ : 2026-03-13 04:03:39
📡 : Topo=scale_free, Mal=0.3, Def=0.2

############################################################
📡 Topo: scale_free | Mal: 0.3 | Def: 0.2 | Seed: 1
############################################################
🚀 Running: MAB
🚀 [System] Loading all client datasets directly into VRAM...
📸 [System] Extracting real validation data for S_Z Probe...
✅ Probe successfully extracted, shape: torch.Size([16, 3, 32, 32])

--- Round 1/15 ---
  [Attack Info] Estimated Benign Update Norm: 9.2459

--- Round 2/15 ---
  [Attack Info] Estimated Benign Update Norm: 7.0396

--- Round 3/15 ---
  [Attack Info] Estimated Benign Update Norm: 6.3303

--- Round 4/15 ---
  [Attack Info] Estimated Benign Update Norm: 5.7207

--- Round 5/15 ---
  [Attack Info] Estimated Benign Update Norm: 5.3302

--- Round 6/15 ---
  [Attack Info] Estimated Benign Update Norm: 4.5530

--- Round 7/15 ---
  [Attack Info] Estimated Benign Update Norm: 4.1549

--- Round 8/15 ---
  [Attack Info] Estimated Ben